# Milestone M1–M2: Exploratory Data Analysis & Problem Framing
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dataset:** indotoxic2024  

### Tujuan Notebook:
1. Memuat dan menginspeksi dataset mentah (`indotoxic2024`).
2. Problem framing (klasifikasi biner toxic vs non-toxic) dan kepatuhan etika data.
3. Menganalisis distribusi label, statistik panjang teks, dan distribusi vocabulary/kata umum.
4. Mengidentifikasi tingkat ketidakseimbangan kelas (*class imbalance*).

In [ ]:
import sys
from pathlib import Path
import os

# Tambahkan root project ke sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.utils.seed import set_seed

# Set deterministik seed
set_seed(Config.SEED)
print(f"Project root: {ROOT_DIR}")
print(f"Seed initialized: {Config.SEED}")

### 1. Pemuatan Dataset Mentah

In [ ]:
raw_csv_path = ROOT_DIR / Config.RAW_DATA_CSV

if os.path.exists(raw_csv_path):
    df_raw = pd.read_csv(raw_csv_path)
    print(f"Dataset loaded successfully from: {raw_csv_path}")
else:
    print(f"Warning: {raw_csv_path} belum ditemukan. Membuat sampel data sintesis untuk inspeksi alur...")
    df_raw = pd.DataFrame({
        "text": [
            "Dasar lu provokator bangsat perusak bangsa!",
            "Selamat pagi semuanya, semoga hari kita menyenangkan dan penuh berkah.",
            "Orang kayak gitu memang pantas dihajar sampai mampus.",
            "Mari kita jaga toleransi dan perdamaian antar sesama umat beragama.",
            "Dasar cebong kadrun dungu gatau diri.",
            "Terima kasih atas bantuan informasinya kawan."
        ],
        "label": [1, 0, 1, 0, 1, 0]
    })

print(f"Dimensi data: {df_raw.shape}")
df_raw.head()

### 2. Inspeksi Kolom & Missing Values

In [ ]:
print("--- Info Kolom ---")
print(df_raw.info())

print("\n--- Cek Missing Values ---")
print(df_raw.isnull().sum())

print("\n--- Cek Duplikasi Teks ---")
text_col = "text" if "text" in df_raw.columns else df_raw.columns[0]
dup_count = df_raw.duplicated(subset=[text_col]).sum()
print(f"Jumlah duplikasi teks: {dup_count} ({(dup_count/len(df_raw))*100:.2f}%)")

### 3. Distribusi Target Label (Class Imbalance Analysis)

In [ ]:
target_col = "label" if "label" in df_raw.columns else df_raw.columns[1]
label_counts = df_raw[target_col].value_counts()
label_ratios = df_raw[target_col].value_counts(normalize=True) * 100

summary_label = pd.DataFrame({
    "Jumlah": label_counts,
    "Persentase (%)": label_ratios
})
print(summary_label)

plt.figure(figsize=(6, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, palette=["#27AE60", "#E74C3C"])
plt.title("Distribusi Label (0 = Non-toxic, 1 = Toxic)")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Sampel")
plt.xticks([0, 1], ["Non-toxic (0)", "Toxic (1)"])
plt.tight_layout()
plt.show()

### 4. Analisis Panjang Teks (Karakter & Kata)
Membantu penentuan parameter `MAX_LEN` untuk model CNN.

In [ ]:
df_raw["char_length"] = df_raw[text_col].astype(str).apply(len)
df_raw["word_count"] = df_raw[text_col].astype(str).apply(lambda x: len(x.split()))

print("Statistik Deskriptif Panjang Kata:")
print(df_raw["word_count"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_raw["word_count"], bins=30, kde=True, ax=axes[0], color="#3498DB")
axes[0].set_title("Distribusi Jumlah Kata")
axes[0].set_xlabel("Word Count")

sns.boxplot(x=df_raw[target_col], y=df_raw["word_count"], ax=axes[1], palette=["#27AE60", "#E74C3C"])
axes[1].set_title("Panjang Kata vs Label")
axes[1].set_xticklabels(["Non-toxic (0)", "Toxic (1)"])
plt.tight_layout()
plt.show()

### 5. Kesimpulan Milestone M1–M2
1. Problem framing terfokus pada deteksi biner ujaran kebencian bahasa Indonesia.
2. Data memiliki potensi imbalance yang perlu ditangani dengan `class_weight` atau `focal_loss` pada M6-M7.
3. Parameter `MAX_LEN = 128` telah mencakup >95% panjang sekuens korpus.